
# Script RvW-tool

#### Ontwikkeld door: Wietse Wierks (HDSR), Rob Tijsen (AGV) & Rafi Senden (AGV)  
In opdracht van: Deltaprogramma Centraal Holland binnen het Maatregelenpakket Wateroverlast. 

**DISCLAIMER:**  
Dit is een werkbestand en nog in ontwikkeling. Signaleer je fouten of onduidelijkheden, neem dan contact op via: wietse.wierks@hdsr.nl  

---

## Introductie

Deze tool is ontwikkeld binnen het Deltaprogramma Centraal Holland voor het maatregelenpakket *Wateroverlast – Traject Ruimte voor Water*.  

De tool bestaat uit twee scripts:
- Voorbewerking van data die gebruikt wordt voor de tool;
- De daadwerkelijke tool waarin de RvW wordt berekend in m³ en m² (dit script).

De invoer en  werking van de functies wordt toegelicht via comments en markdown cellen. In de comments in het script zelf wordt een toelichting gegeven over de werking van de functie. In de markdown cellen wordt uitgelegd per stap waarom deze functie wordt uitgevoerd. 


## Workflow

Het script is opgedeeld in:

### Deel 1 - Bouw volume-oppervlakte dataset per peilvak per polder

### Deel 2 - Aggregeer dataset naar polderniveau

### Deel 3 - Bereken RvW o.b.v. Ontwerpbui T


# Deel 1 - Bouw volume-oppervlakte dataset per peilvak per polder

In [9]:
import arcpy
import os
import numpy as np
import pandas as pd
import re
from tqdm.notebook import tqdm
from pathlib import Path
from arcpy.sa import SetNull 
from collections import defaultdict

In [2]:
# Even voor de fatoe!

### 1.1 Bereken volume en inundatie per peilvak

**Doel**  
Bepalen van inundatievolumes, inundatieoppervlaktes en de verdeling hiervan over verschillende landgebruikstypen voor alle peilgebieden binnen een polder bij oplopende waterstanden. Hiervoor wordt gebruikgemaakt van de voorbewerkte AHN-rasterdata die in de Setup-fase per polder is samengesteld. Per peilgebied wordt vanuit het geldende peil de waterstand stapsgewijs verhoogd met een vooraf gedefinieerd interval tot een maximale verhoging. Voor iedere waterstandsstap wordt op basis van het AHN bepaald welke delen van het maaiveld overstromen, waarna het inundatieoppervlak en het waterbergingsvolume worden berekend en uitgesplitst naar slootoppervlak en de verschillende landgebruikstypen.

**Output**
- Resultatentabel met inundatievolume en inundatieoppervlak per peilgebied en waterstand
- Oppervlaktes en volumes uitgesplitst naar landgebruiksklasse
- CSV-bestand met tussentijdse resultaten


### 1.1 Achtergrondfunctie: Bereken volume en inundatie per x cm peilstijging

Deze achtergrondfunctie berekent per peilvak in iedere polder o.b.v. de voorbewerkte AHN-raster data stapsgewijs het volume en inundatie op het maaiveld per mogelijke waterstand.


In [21]:
def maak_veilige_naam(
    naam,
    *,
    target="gdb_object",   # "gdb_object" of "filesystem"
    workspace=None,
    max_len=70
):
    """
    Maakt een veilige naam voor:
    - target="filesystem"  → mappen + .gdb namen
    - target="gdb_object"  → feature classes / tabellen in een GDB

    Zet een tekst om naar een veilige naam voor gebruik in het
    bestandssysteem of binnen een geodatabase.
    Bewerkingen:
        - omzetting naar kleine letters
        - spaties vervangen door underscores
        - ongeldige tekens verwijderen/vervangen
        - voorkomen dat namen met een cijfer beginnen
        - validatie van GDB-objectnamen via arcpy
    """
    s = str(naam).strip().lower()

    # Uniforme normalisatie
    s = s.replace(" ", "_")
    s = re.sub(r"[^\w]", "_", s)   # ook - / \ etc.
    s = re.sub(r"_+", "_", s)

    # Niet beginnen met cijfer
    if s and s[0].isdigit():
        s = f"p_{s}"

    if target == "filesystem":
        return s.strip("_")

    if target == "gdb_object":
        if workspace is None:
            raise ValueError("workspace is verplicht bij target='gdb_object'")

        s = s[:max_len]
        s = s.strip("_")
        return arcpy.ValidateTableName(s, workspace)

    raise ValueError(f"Onbekend target: {target}")
    
def hypsometrie(elevaties, wls):
    z = np.sort(elevaties)
    csum = np.concatenate(([0.0], np.cumsum(z)))
    aantal_cellen = np.searchsorted(z, wls, side="left")
    diepte_som = wls * aantal_cellen - csum[aantal_cellen]
    
    return aantal_cellen, diepte_som
    
def bereken_volumes_fast(
    root_folder,
    fld_waterschap="Waterschap",
    fld_polder="Naam_1",
    fld_peilgebied="CODE",
    fld_maxpeil="Peil_zo",
    fld_maalpand="Type",
    fld_op_pg="OPP_PG",
    fld_op_pldr="OPP_PLDR",
    peil_type="zo",
    interval=0.02,
    hoogte_x=0.5,
    select_waterschap=None,
    select_polder=None
):
    """
    Berekent inundatievolumes en overstroomde oppervlaktes voor alle
    peilgebieden binnen geselecteerde polders en waterschappen op basis
    van hoogte- en landgebruiksrasters.

    Voor ieder peilgebied wordt de waterstand stapsgewijs verhoogd vanaf
    het opgegeven peil tot een maximale verhoging. Per waterstand worden
    inundatiediepte, inundatieoppervlak en waterbergingsvolume berekend.
    Daarnaast wordt onderscheid gemaakt naar verschillende typen
    landgebruik.

    Parameters
    ----------
    root_folder : str
        Hoofdmap met waterschappen en polders. Per polder worden een
        geodatabase, hoogtebestand en landgebruiksraster verwacht.

    fld_waterschap : str, default "Waterschap"
        Naam van het veld met de waterschapsnaam.

    fld_polder : str, default "Naam_1"
        Naam van het veld met de poldernaam.

    fld_peilgebied : str, default "CODE"
        Unieke identificatie van het peilgebied.

    fld_maxpeil : str, default "Peil_zo"
        Veld met het uitgangspeil waarop de berekeningen starten.

    fld_maalpand : str, default "Type"
        Veld met het type maalpand of afwateringseenheid.

    fld_op_pg : str, default "OPP_PG"
        Veld met de oppervlakte van het peilgebied.

    fld_op_pldr : str, default "OPP_PLDR"
        Veld met de oppervlakte van de polder.

    peil_type : str, default "zo"
        Type peil dat wordt gebruikt bij het selecteren van het
        hoogteraster.

    interval : float, default 0.02
        Waterstandsverhoging per stap in meters.

    hoogte_x : float, default 0.5
        Maximale verhoging van de waterstand boven het uitgangspeil.

    select_waterschap : str, optional
        Indien opgegeven worden uitsluitend polders van dit
        waterschap verwerkt.

    select_polder : str, optional
        Indien opgegeven wordt uitsluitend deze polder verwerkt.

    Returns
    -------
    pandas.DataFrame

    DataFrame met per peilgebied en waterstand:

    - waterschap
    - polder
    - oppervlakte polder
    - peilgebied
    - oppervlakte peilgebied
    - peiltype
    - maalpandtype
    - waterstand (WL)
    - inundatievolume (m³)
    - inundatieoppervlak (m²)

    Daarnaast worden voor de volgende landgebruiksklassen zowel
    oppervlaktes als volumes bepaald:

    - water
    - grasland
    - akker
    - hoogwaardig land -en tuinbouw (afgekort: tuinbouw)
    - bebouwing

    Werkwijze
    ---------
    1. Doorloop alle waterschappen en polders binnen de hoofdmap.
    2. Lees de peilgebiedgeometrie en bijbehorende attributen in.
    3. Lees hoogte- en landgebruiksrasters in.
    4. Clip rasters naar het rekengebied van de polder.
    5. Converteer de rasters naar NumPy-arrays voor snelle verwerking.
    6. Rasteriseer de peilgebieden zodat per cel bekend is bij welk
       peilgebied deze hoort.
    7. Verhoog de waterstand stapsgewijs vanaf het uitgangspeil.
    8. Bereken voor iedere stap:
       - waterdiepte per rastercel
       - totaal inundatievolume
       - totaal inundatieoppervlak
       - oppervlaktes per landgebruikstype
       - volumes per landgebruikstype
    9. Sla tussentijdse resultaten op naar CSV.
    10. Retourneer alle resultaten als pandas DataFrame.

    Opmerkingen
    -----------
    De berekeningen maken gebruik van NumPy-arrays in plaats van
    herhaalde ArcGIS-rasteranalyses per peilgebied. Hierdoor kunnen
    grote aantallen waterstandsberekeningen aanzienlijk sneller worden
    uitgevoerd dan met een volledig ArcGIS-gebaseerde workflow.
    """ 
    
    arcpy.env.addOutputsToMap = False
    arcpy.env.overwriteOutput = True
    arcpy.env.parallelProcessingFactor = "100%"
    
    arcpy.CheckOutExtension("Spatial")

    rd_new = arcpy.SpatialReference(28992)
    arcpy.env.outputCoordinateSystem = rd_new

    resultaten = []
    
    output_csv = os.path.join(root_folder, "tussenresultaten.csv")

    for ws in tqdm(os.listdir(root_folder), desc="Waterschappen"):

        if select_waterschap:
            if maak_veilige_naam(ws, target="filesystem") != \
               maak_veilige_naam(select_waterschap, target="filesystem"):
                continue

        ws_path = os.path.join(root_folder, ws)
        if not os.path.isdir(ws_path):
            continue

        for polder in tqdm(os.listdir(ws_path), desc=f"Polders {ws}", leave=False):

#             try:

                if select_polder:
                    if maak_veilige_naam(polder, target="filesystem") != \
                       maak_veilige_naam(select_polder, target="filesystem"):
                        continue

                polder_path = os.path.join(ws_path, polder)
                if not os.path.isdir(polder_path):
                    continue

                print(f"--- {ws} / {polder} ---")

                ws_norm = maak_veilige_naam(ws, target="filesystem")
                polder_norm = maak_veilige_naam(polder, target="filesystem")

                # --------------------------------------------------
                # Rekengebied
                # --------------------------------------------------
                gdb = os.path.join(polder_path, f"{polder}.gdb")
                fc = os.path.join(gdb, f"rekengebied_{polder}")

                if not arcpy.Exists(fc):
                    print("--- geen rekengebied! ---")
                    continue

                fields = [
                    fld_waterschap,
                    fld_polder,
                    fld_peilgebied,
                    fld_maxpeil,
                    fld_maalpand,
                    fld_op_pg,
                    fld_op_pldr
                ]

                rows = list(arcpy.da.SearchCursor(fc, fields))
                if not rows:
                    continue

                df = pd.DataFrame(rows, columns=[
                    "waterschap", "polder", "peilgebied",
                    "maxpeil", "maalpand",
                    "opp_pg", "opp_pldr"
                ])


                print(f"  polder map naam: {polder}")
                print(f"  polder veld (uniek): {df['polder'].unique()}")

                df["polder_norm"] = df["polder"].apply(
                    lambda x: maak_veilige_naam(x, target="filesystem")
                )

                polder_norm_check = maak_veilige_naam(polder, target="filesystem")

                print(f"  polder_norm map: {polder_norm_check}")
                print(f"  polder_norm df (uniek): {df['polder_norm'].unique()}")

                df = df[df["polder_norm"] == polder_norm_check]

                print(f"  df lengte na filter: {len(df)}")

                if df.empty:
                    print("--- dataframe leeg -> skip!!! ---")
                    continue

                if df.empty:
                    continue

                # --------------------------------------------------
                # Rasters (met DEBUG)
                # --------------------------------------------------
                hoogte_raster = os.path.join(
                    polder_path,
                    f"{ws_norm}_{polder_norm}_Peil_{peil_type}.tif"
                )

                lu_raster = os.path.join(
                    polder_path,
                    f"{ws_norm}_{polder_norm}_lu.tif"
                )

                print(f"  hoogte raster: {hoogte_raster}")
                print(f"  landuse raster: {lu_raster}")

                if not os.path.exists(hoogte_raster):
                    print("--- hoogte raster ontbreekt! ---")
                if not os.path.exists(lu_raster):
                    print("--- landuse raster ontbreekt! ---")

                if not (os.path.exists(hoogte_raster) and os.path.exists(lu_raster)):
                    print("--- skip polder! ---")
                    continue
                else:
                    print("--- beide rasters gevonden ---")

                print("--- start berekening ---")

                # --------------------------------------------------
                # Hoogte raster laden (GEEN ProjectRaster tenzij nodig)
                # --------------------------------------------------
                print("1 Raster openen")
                ahn = arcpy.Raster(hoogte_raster)
                
                print("2 CRS check")
                if ahn.spatialReference.factoryCode != 28992:
                    out_raster = os.path.join(
                        arcpy.env.scratchGDB,
                        f"ahn_{polder_norm}"
                    )

                    ahn = arcpy.management.ProjectRaster(
                        ahn,
                        out_raster,
                        rd_new,
                        "BILINEAR"
                    )

                ahn = arcpy.Raster(ahn)

                cellsize = ahn.meanCellWidth
                cellarea = cellsize ** 2

                # --------------------------------------------------
                # Clip 1x op rekengebied
                # --------------------------------------------------
                layer = f"lyr_{polder_norm}"
                arcpy.MakeFeatureLayer_management(fc, layer)
                
                print("4 Set environments")
                arcpy.env.snapRaster = ahn
                arcpy.env.extent = layer
                arcpy.env.cellSize = ahn
                
                print("5 ExtractByMask START")
                ahn_clip = arcpy.sa.ExtractByMask(ahn, layer)
                
                print("6 ExtractByMask GEREED")
                ahn_clip = arcpy.Raster(ahn_clip)
                
                print("7 Raster info")
                print("width =", ahn_clip.width)
                print("height =", ahn_clip.height)
                
                # NumPy conversie
                pixel_type = ahn_clip.pixelType
                is_integer = not pixel_type.startswith("F")
                
                print("8 RasterToNumPyArray START")

                NODATA = -99999
                use_tiles = False

                try:

                    ahn_arr = arcpy.RasterToNumPyArray(
                        ahn_clip,
                        nodata_to_value=NODATA
                    ).astype(np.float32)

                    ahn_arr[ahn_arr == NODATA] = np.nan

                    if is_integer:
                        # mm -> m
                        ahn_arr /= 1000.0

                    mask_arr = ~np.isnan(ahn_arr)

                    print("9 RasterToNumPyArray GEREED")

                    # --------------------------------------------------
                    # Landuse
                    # --------------------------------------------------
                    lu = arcpy.Raster(lu_raster)

                    lu_clip = arcpy.sa.ExtractByMask(
                        lu,
                        layer
                    )

                    lu_arr = arcpy.RasterToNumPyArray(
                        lu_clip,
                        nodata_to_value=-1
                    ).astype(np.int16)

                except Exception as e:

                    if "pixel block exceeds the maximum size allowed" not in str(e).lower():
                        raise

                    print("--- Raster te groot ---")
                    print("--- Overschakelen naar tile processing ---")

                    use_tiles = True

                    # --------------------------------------------------
                    # Landuse
                    # --------------------------------------------------
                    lu = arcpy.Raster(lu_raster)

                    lu_clip = arcpy.sa.ExtractByMask(
                        lu,
                        layer
                    )

                    # --------------------------------------------------
                    # Raster opdelen in 4 tiles
                    # --------------------------------------------------
                    MAX_PIXELS = 100_000_000

                    n_tiles_x = 1
                    n_tiles_y = 1

                    while True:

                        tile_width = int(
                            np.ceil(ahn_clip.width / n_tiles_x)
                        )

                        tile_height = int(
                            np.ceil(ahn_clip.height / n_tiles_y)
                        )

                        n_pixels_tile = tile_width * tile_height

                        if n_pixels_tile <= MAX_PIXELS:
                            break

                        n_tiles_x += 1
                        n_tiles_y += 1

                    print(
                        f"{n_tiles_x} x {n_tiles_y} tiles"
                    )

                    print(
                        f"tile grootte = "
                        f"{tile_width} x {tile_height}"
                    )

                    print(
                        f"pixels per tile = "
                        f"{n_pixels_tile:,}"
                    )

                    tile_width = int(
                        np.ceil(ahn_clip.width / n_tiles_x)
                    )

                    tile_height = int(
                        np.ceil(ahn_clip.height / n_tiles_y)
                    )

                    print(
                        f"Raster: {ahn_clip.width} x {ahn_clip.height}"
                    )

                    print(
                        f"Tile grootte: "
                        f"{tile_width} x {tile_height}"
                    )

                    # deze worden later gevuld
                    pg_elevaties = defaultdict(list)

                    lu_elevaties = {
                        0: defaultdict(list),
                        1: defaultdict(list),
                        2: defaultdict(list),
                        3: defaultdict(list),
                        4: defaultdict(list)
                    }

                # --------------------------------------------------
                # peilgebied raster
                # --------------------------------------------------
                # Zorg dat peilgebied numeriek is
                if fld_peilgebied not in [f.name for f in arcpy.ListFields(fc) if f.type in ("Integer", "SmallInteger")]:
                    if "pg_id" not in [f.name for f in arcpy.ListFields(fc)]:
                        arcpy.AddField_management(fc, "pg_id", "LONG")
                        arcpy.CalculateField_management(fc, "pg_id", "!OBJECTID!", "PYTHON3")
                    fld_pg = "pg_id"
                    df["pg_id"] = df.index + 1  # fallback mapping
                else:
                    fld_pg = fld_peilgebied

                pg_raster = os.path.join(arcpy.env.scratchGDB, f"pg_{polder_norm}")

                arcpy.env.snapRaster = ahn_clip
                arcpy.env.extent = ahn_clip
                arcpy.env.cellSize = ahn_clip

                pg_raster = arcpy.conversion.PolygonToRaster(
                    fc,
                    fld_pg,
                    pg_raster,
                    cell_assignment="MAXIMUM_AREA",
                    priority_field=fld_pg,
                    cellsize=cellsize
                )

                pg_raster = arcpy.Raster(pg_raster)
                
                if not use_tiles:    
                    
                    pg_arr = arcpy.RasterToNumPyArray(pg_raster)
                    
                if use_tiles:

                    for row0 in range(
                        0,
                        ahn_clip.height,
                        tile_height
                    ):

                        nrows = min(
                            tile_height,
                            ahn_clip.height - row0
                        )

                        for col0 in range(
                            0,
                            ahn_clip.width,
                            tile_width
                        ):

                            ncols = min(
                                tile_width,
                                ahn_clip.width - col0
                            )

                            ll = arcpy.Point(
                                ahn_clip.extent.XMin +
                                col0 * cellsize,

                                ahn_clip.extent.YMin +
                                row0 * cellsize
                            )

                            print(
                                f"Tile row={row0}, col={col0}"
                            )

                            ahn_tile = arcpy.RasterToNumPyArray(
                                ahn_clip,
                                lower_left_corner=ll,
                                ncols=ncols,
                                nrows=nrows,
                                nodata_to_value=NODATA
                            ).astype(np.float32)

                            lu_tile = arcpy.RasterToNumPyArray(
                                lu_clip,
                                lower_left_corner=ll,
                                ncols=ncols,
                                nrows=nrows,
                                nodata_to_value=-1
                            ).astype(np.int16)

                            pg_tile = arcpy.RasterToNumPyArray(
                                pg_raster,
                                lower_left_corner=ll,
                                ncols=ncols,
                                nrows=nrows,
                                nodata_to_value=-1
                            )

                            ahn_tile[ahn_tile == NODATA] = np.nan

                            if is_integer:
                                ahn_tile /= 1000.0

                            for pg in np.unique(pg_tile):

                                if pg <= 0:
                                    continue

                                mask_pg = (pg_tile == pg)

                                vals = ahn_tile[mask_pg]
                                vals = vals[~np.isnan(vals)]

                                if len(vals):
                                    pg_elevaties[pg].append(vals)

                                for lu_code in [0, 1, 2, 3, 4]:

                                    vals_lu = ahn_tile[
                                        mask_pg &
                                        (lu_tile == lu_code)
                                    ]

                                    vals_lu = vals_lu[
                                        ~np.isnan(vals_lu)
                                    ]

                                    if len(vals_lu):
                                        lu_elevaties[lu_code][pg].append(
                                            vals_lu
                                        )

                            del ahn_tile
                            del lu_tile
                            del pg_tile

                    for pg in pg_elevaties:

                        if len(pg_elevaties[pg]):

                            pg_elevaties[pg] = np.concatenate(
                                pg_elevaties[pg]
                            )

                        else:

                            pg_elevaties[pg] = np.array(
                                [],
                                dtype=np.float32
                            )

                    for lu_code in [0, 1, 2, 3, 4]:

                        for pg in lu_elevaties:
                            if len(lu_elevaties[lu_code][pg]):

                                lu_elevaties[lu_code][pg] = np.concatenate(
                                    lu_elevaties[lu_code][pg]
                                )

                            else:

                                lu_elevaties[lu_code][pg] = np.array(
                                    [],
                                    dtype=np.float32
                                )
                                
                arcpy.Delete_management(layer)

                # --------------------------------------------------
                # Berekeningen
                # --------------------------------------------------
                for _, row in tqdm(df.iterrows(), total=len(df), leave=False):

                    pg_val = row[fld_peilgebied] if fld_pg == fld_peilgebied else row["pg_id"]
                    
                    if use_tiles:

                        elevaties_pg = pg_elevaties.get(
                            pg_val,
                            np.array([], dtype=np.float32)
                        )
                    
                    else:
                        mask_pg = (pg_arr == pg_val)
   
                        elevaties_pg = ahn_arr[mask_pg]
                        elevaties_pg = elevaties_pg[~np.isnan(elevaties_pg)] 
    
                    lu_aantal_cellen = [0,0,0,0,0]
                    lu_dieptesommen = [0,0,0,0,0]
                    
                    wls = np.arange(row["maxpeil"],row["maxpeil"] + hoogte_x + interval,interval)

                    for lu in [0, 1, 2, 3, 4]:
                        
                        if use_tiles:
                            lu_pg = lu_elevaties[lu].get(pg_val, np.array([], dtype=np.float32))
                            print(
                            f"LU={lu}",
                            type(lu_pg),
                            len(lu_pg)
                        )   
                            
                        else:   
                            lu_pg = ahn_arr[mask_pg & (lu_arr == lu)]
                            lu_pg = lu_pg[~np.isnan(lu_pg)]
                        lu_aantal_cellen[lu], lu_dieptesommen[lu] = hypsometrie(lu_pg, wls)

                    aantal_cellen, dieptesommen = hypsometrie(elevaties_pg, wls)

                    for i, WL in enumerate(wls):
                        resultaten.append([
                            row["waterschap"],
                            row["polder"],
                            row["opp_pldr"],
                            row["peilgebied"],
                            row["opp_pg"],
                            peil_type,
                            row["maalpand"],
                            WL,
                            
                            dieptesommen[i] * cellarea,
                            aantal_cellen[i] * cellarea,
                            
                            lu_aantal_cellen[0][i] * cellarea, # water
                            lu_aantal_cellen[1][i] * cellarea, # grasland
                            lu_aantal_cellen[2][i] * cellarea, # akker
                            lu_aantal_cellen[3][i] * cellarea, # hoogwaardig land -en tuinbouw 
                            lu_aantal_cellen[4][i] * cellarea, # bebouwing
                            
                            lu_dieptesommen[0][i] * cellarea, # water volume
                            lu_dieptesommen[1][i] * cellarea, # grasland
                            lu_dieptesommen[2][i] * cellarea, # akker
                            lu_dieptesommen[3][i] * cellarea, # hoogwaarding land -en tuinbouw
                            lu_dieptesommen[4][i] * cellarea # bebouwing
                        ])

                # tussentijds opslaan
                pd.DataFrame(
                    resultaten,
                    columns=[
                        "waterschap", "polder", "opp_pldr_m2",
                        "peilgebied", "opp_pg_m2", "peil_type",
                        "maalpand", "WL",
                        "volume_m3", "inundatie_m2",
                        # oppervlaktes
                        "water_m2", "grasland_m2", "akker_m2",
                        "tuinbouw_m2", "bebouwing_m2",
                        # volumes 
                        "water_m3", "grasland_m3", "akker_m3",
                        "tuinbouw_m3", "bebouwing_m3"
                    ]
                ).to_csv(output_csv, index=False)

    return pd.DataFrame(
        resultaten,
        columns=[
            "waterschap",
            "polder",
            "opp_pldr_m2",
            "peilgebied",
            "opp_pg_m2",
            "peil_type",
            "maalpand",
            "WL",
            "volume_m3",
            "inundatie_m2",
            "water_m2",
            "grasland_m2",
            "akker_m2",
            "tuinbouw_m2",   
            "bebouwing_m2",
            "water_m3",
            "grasland_m3",
            "akker_m3",
            "tuinbouw_m3",
            "bebouwing_m3"
        ]
    )


### 1.1 Invoer: Bereken volume en inundatie per x cm peilstijging

**Doel**
Bepalen van inundatievolumes, inundatieoppervlaktes en de verdeling hiervan over verschillende landgebruikstypen voor alle peilgebieden binnen geslecteerde polders.

**Configuratie**
- Geef root_folder aan waar rekengebied mappen zich bevinden, eventueel specificeren tot één waterschap én ook nog polder
- Kies te gebruiken peiltype (mits deze te vinden is in rekengebied attributetable), bijv. zomerpeil (e.g. Peil_zo)
- Instellen van waterstandstap (interval)
- Instellen van maximale waterstandsverhoging boven uitgangspeil
- Koppeling van benodigde velden voor peilgebieden, peil en oppervlaktes.

**Voorbeeldconfiguratie**
- Waterschap: HHNK
- Polder: Eilandspolder
- Peiltype: zomerpeil (Peil_zo)
- Waterstandsinterval: 0,01 m (1 cm)
- Maximale waterstandsverhoging: 1,0 m boven het uitgangspeil

**Output**

- DataFrame met per peilgebied en waterstand:
    - inundatievolume (m³)
    - inundatieoppervlak (m²)
    - waterstand (m NAP)
    - oppervlakte van peilgebied en polder
    - type maalpand
- Uitsplitsing van inundatieoppervlaktes en volumes naar:
    - water
    - grasland
    - akkerbouw
    - hoogwaardig land- en tuinbouw
    - bebouwing
    - CSV-bestand met tussentijdse resultaten voor verdere analyse of controle


In [22]:
root_folder = r"D:\04_results\26072026"

df = bereken_volumes_fast(
    root_folder=root_folder,
    fld_waterschap="Waterschap",
    fld_polder="Naam_12",
    fld_peilgebied="CODE",
    fld_maxpeil="Peil_zo",
    fld_maalpand="Type",
    fld_op_pg="OPP_PG",
    fld_op_pldr="OPP_PLDR",

    peil_type="zo",
    interval=0.01,
    hoogte_x=3, # even kijken wat de hoogte is misschien naar 50 cm kan

    # optioneel:
    select_waterschap="Rijnland",
    select_polder="Haarlemmermeerpolder"
)

df

# checken welke polders er in je dataframe zitten
# for p in df["polder"].unique():
#     print(p)


Waterschappen:   0%|          | 0/50 [00:00<?, ?it/s]

Polders rijnland:   0%|          | 0/205 [00:00<?, ?it/s]

--- rijnland / haarlemmermeerpolder ---
  polder map naam: haarlemmermeerpolder
  polder veld (uniek): ['Haarlemmermeerpolder']
  polder_norm map: haarlemmermeerpolder
  polder_norm df (uniek): ['haarlemmermeerpolder']
  df lengte na filter: 51
  hoogte raster: D:\04_results\26072026\rijnland\haarlemmermeerpolder\rijnland_haarlemmermeerpolder_Peil_zo.tif
  landuse raster: D:\04_results\26072026\rijnland\haarlemmermeerpolder\rijnland_haarlemmermeerpolder_lu.tif
--- beide rasters gevonden ---
--- start berekening ---
1 Raster openen
2 CRS check
4 Set environments
5 ExtractByMask START
6 ExtractByMask GEREED
7 Raster info
width = 36679
height = 37918
8 RasterToNumPyArray START
--- Raster te groot ---
--- Overschakelen naar tile processing ---
4 x 4 tiles
tile grootte = 9170 x 9480
pixels per tile = 86,931,600
Raster: 36679 x 37918
Tile grootte: 9170 x 9480
Tile row=0, col=0
Tile row=0, col=9170
Tile row=0, col=18340
Tile row=0, col=27510
Tile row=9480, col=0
Tile row=9480, col=9170
Tile r

  0%|          | 0/51 [00:00<?, ?it/s]

LU=0 <class 'numpy.ndarray'> 1108277
LU=1 <class 'numpy.ndarray'> 3697413
LU=2 <class 'numpy.ndarray'> 9825535
LU=3 <class 'numpy.ndarray'> 19119
LU=4 <class 'numpy.ndarray'> 2903664
LU=0 <class 'numpy.ndarray'> 118190
LU=1 <class 'numpy.ndarray'> 159265
LU=2 <class 'numpy.ndarray'> 0
LU=3 <class 'numpy.ndarray'> 96309
LU=4 <class 'numpy.ndarray'> 305197
LU=0 <class 'numpy.ndarray'> 40611
LU=1 <class 'numpy.ndarray'> 48039
LU=2 <class 'numpy.ndarray'> 0
LU=3 <class 'numpy.ndarray'> 0
LU=4 <class 'numpy.ndarray'> 10689
LU=0 <class 'numpy.ndarray'> 254292
LU=1 <class 'numpy.ndarray'> 844497
LU=2 <class 'numpy.ndarray'> 560891
LU=3 <class 'numpy.ndarray'> 1778360
LU=4 <class 'numpy.ndarray'> 2343024
LU=0 <class 'list'> 12


ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (12,) + inhomogeneous part.

### Tussentijdse stap: sla gegenereerde dataset op in drievoud (.pkl, .txt & .csv)

**Doel**
Opslaan van de berekende inundatieresultaten in verschillende bestandsformaten voor verdere analyse, uitwisseling en hergebruik. 


**Werking**
- Aanmaken van de uitvoermap indien deze nog niet bestaat
- Wegschrijven van de resultaatdataset naar meerdere bestandsformaten

**Output**
- PKL-bestand (.pkl): Python-native opslagformaat voor snel hergebruik in analysescripts
- TXT-bestand (.txt): tab-gescheiden tekstbestand, geschikt voor gebruik in ArcGIS en andere GIS-software
- CSV-bestand (.csv): algemeen uitwisselingsformaat voor Excel, Power BI en andere analysetools 

**Resultaat**:
De volledige resultatentabel wordt opgeslagen in meerdere formaten, zodat deze zowel binnen Python als in GIS- en rapportagetools eenvoudig kan worden gebruikt.

In [9]:
import os

output_folder = r"D:\04_results"
os.makedirs(output_folder, exist_ok=True)

# basisnaam
base_name = "df_rvw_gav" #bepaal zelf welke bestandsnaam

# Pickle (voor Python)
df.to_pickle(
    os.path.join(output_folder, f"{base_name}.pkl")
)

# TXT (tab-separated, handig voor ArcGIS)
df.to_csv(
    os.path.join(output_folder, f"{base_name}.txt"),
    sep="\t",
    index=False
)

# CSV (voor Excel / delen)
df.to_csv(
    os.path.join(output_folder, f"{base_name}.csv"),
    index=False
)

# Deel 2 - Aggregeer dataset naar polderniveau

### 2.1 Inladen van resultaten in dataframe

Voordat dataset naar polderniveau geaggregeerd wordt, moet de resulterende dataset uit deel 1 opnieuw worden ingeladen. Dat kan hieronder:

In [83]:
df_agv = pd.read_csv(
    r"D:\04_results\agv_results\df_rvw_agv.csv")
    
df_hdsr = pd.read_csv(
    r"D:\04_results\hdsr_results\df_rvw_hdsr.csv")
    
df_hhr = pd.read_csv(
    r"D:\04_results\hhr_results\df_rvw_hhr.csv")

df_hhnk = pd.read_csv(r"D:\04_results\hhnk_results\df_rvw_hhnk.csv")

# laat begin en eind of gehele dataframe zien
# pd.set_option("display.max_rows", None)
# pd.reset_option("display.max_rows")


### 2.2 Aggregeer dataframe naar polderniveau

In dit deel wordt de dataframe naar polderniveau geaggregeerd.


### 2.2 Achtergrondfunctie: aggregeer dataframe naar polderniveau

**Doel**
- Samenvoegen van inundatieresultaten op peilgebiedniveau tot één consistente waterstandsreeks per polder.

**Werking**
- Groeperen van resultaten per waterschap en polder.
- Samenvoegen van inundatievolumes, inundatieoppervlaktes en landgebruiksstatistieken van alle peilgebieden.
- Vastleggen van de geaggregeerde resultaten per waterstand.

**Output**
- Inundatievolume per polder en waterstand.
- Inundatieoppervlak per polder en waterstand.
- Uitsplitsing naar landgebruikstype.


In [80]:
# def aggregate_to_polder_level(df):
    
#     """
#     Aggregeert inundatieresultaten van peilgebiedniveau naar polderniveau.

#     Voor iedere polder wordt per waterstand (WL) het totale inundatievolume,
#     inundatieoppervlak en de verdeling naar landgebruik bepaald. Hierbij
#     worden de resultaten van alle peilgebieden binnen dezelfde polder
#     samengevoegd.

#     Returns
#     -------
#     pandas.DataFrame

#     DataFrame met per polder en waterstand:

#     - waterschap
#     - polder
#     - waterstand (WL)
#     - oppervlakte polder
#     - totale oppervlakte peilgebieden
#     - inundatievolume (m³)
#     - inundatieoppervlak (m²)

#     Daarnaast worden voor de volgende landgebruiksklassen zowel
#     oppervlaktes als volumes bepaald:

#     - water
#     - grasland
#     - akker
#     - hoogwaardig land- en tuinbouw (afgekort: tuinbouw)
#     - bebouwing

#     Werkwijze
#     ---------
#     1. Groepeer de invoergegevens per waterschap en polder.
#     2. Bepaal per polder het bereik van waterstanden.
#     3. Doorloop iedere waterstand binnen dit bereik.
#     4. Selecteer per peilgebied de laatst beschikbare berekening die
#        kleiner dan of gelijk is aan de huidige waterstand.
#     5. Tel volumes, inundatieoppervlaktes en landgebruiksstatistieken
#        van alle peilgebieden binnen de polder bij elkaar op.
#     6. Sla de geaggregeerde resultaten op in een nieuwe DataFrame.
#     7. Retourneer de resultaten op polderniveau.

#     Opmerkingen
#     -----------
#     Omdat peilgebieden verschillende uitgangspeilen kunnen hebben,
#     wordt per waterstand steeds de laatst beschikbare berekening van
#     ieder peilgebied gebruikt. Hierdoor ontstaat een consistente
#     waterstandsreeks op polderniveau.
#     """
#     import numpy as np
#     df = df.copy()

#     df["waterschap"] = df["waterschap"].astype(str).str.strip()
#     df["polder"] = df["polder"].astype(str).str.strip()
#     df["WL"] = df["WL"].astype(float)

#     result_rows = []
#     sum_cols    = ["volume_m3", "inundatie_m2", "water_m2", "grasland_m2", "akker_m2", "tuinbouw_m2", "bebouwing_m2", 
#                   "water_m3", "grasland_m3", "akker_m3", "tuinbouw_m3", "bebouwing_m3", "opp_pg_m2"]

#     # per polder
#     for (ws, polder), df_pol in df.groupby(["waterschap", "polder"]):

#         # globale WL range
#         wl_min = df_pol["WL"].min()
#         wl_max = df_pol["WL"].max()
        
#         WL_range = np.arange(wl_min, wl_max + 0.001, 0.02)
#         WL_df    = pd.DataFrame({"WL": WL_range})
        
#         summed = None
        
#         # loop over peilgebieden
#         for pg, df_pg in df_pol.groupby("peilgebied"):
#             df_pg = df_pg.sort_values("WL")
#             merged = pd.merge_asof(WL_df,df_pg[["WL"] + sum_cols],on="WL",direction="backward")[sum_cols].fillna(0.0)
            
#             summed = merged if summed is None else summed.add(merged)
        
#         print(summed)
        
#         result_rows.append([
#             ws, polder, WL_range, df_pol["opp_pldr_m2"].iloc[0], summed
#         ])
        

# #     df_out = pd.DataFrame(result_rows, columns=[
# #         "waterschap",
# #         "polder",
# #         "WL",
# #         "opp_pldr_m2",
# #         "opp_pg_m2",
# #         "volume_m3",
# #         "inundatie_m2",
# #         "water_m2",
# #         "grasland_m2",
# #         "akker_m2",
# #         "tuinbouw_m2",
# #         "bebouwing_m2",
# #         "water_m3",
# #         "grasland_m3",
# #         "akker_m3",
# #         "tuinbouw_m3",
# #         "bebouwing_m3"
# #     ])

# #     return df_out


In [32]:
def aggregate_to_polder_level(df):
    
    """
    Aggregeert inundatieresultaten van peilgebiedniveau naar polderniveau.

    Voor iedere polder wordt per waterstand (WL) het totale inundatievolume,
    inundatieoppervlak en de verdeling naar landgebruik bepaald. Hierbij
    worden de resultaten van alle peilgebieden binnen dezelfde polder
    samengevoegd.

    Returns
    -------
    pandas.DataFrame

    DataFrame met per polder en waterstand:

    - waterschap
    - polder
    - waterstand (WL)
    - oppervlakte polder
    - totale oppervlakte peilgebieden
    - inundatievolume (m³)
    - inundatieoppervlak (m²)

    Daarnaast worden voor de volgende landgebruiksklassen zowel
    oppervlaktes als volumes bepaald:

    - water
    - grasland
    - akker
    - hoogwaardig land- en tuinbouw (afgekort: tuinbouw)
    - bebouwing

    Werkwijze
    ---------
    1. Groepeer de invoergegevens per waterschap en polder.
    2. Bepaal per polder het bereik van waterstanden.
    3. Doorloop iedere waterstand binnen dit bereik.
    4. Selecteer per peilgebied de laatst beschikbare berekening die
       kleiner dan of gelijk is aan de huidige waterstand.
    5. Tel volumes, inundatieoppervlaktes en landgebruiksstatistieken
       van alle peilgebieden binnen de polder bij elkaar op.
    6. Sla de geaggregeerde resultaten op in een nieuwe DataFrame.
    7. Retourneer de resultaten op polderniveau.

    Opmerkingen
    -----------
    Omdat peilgebieden verschillende uitgangspeilen kunnen hebben,
    wordt per waterstand steeds de laatst beschikbare berekening van
    ieder peilgebied gebruikt. Hierdoor ontstaat een consistente
    waterstandsreeks op polderniveau.
    """
    import numpy as np
    df = df.copy()

    df["waterschap"] = df["waterschap"].astype(str).str.strip()
    df["polder"] = df["polder"].astype(str).str.strip()
    df["WL"] = df["WL"].astype(float)

    result_rows = []

    # per polder
    for (ws, polder), df_pol in df.groupby(["waterschap", "polder"]):

        # globale WL range
        wl_min = df_pol["WL"].min()
        wl_max = df_pol["WL"].max()

        WL_range = np.arange(wl_min, wl_max + 0.001, 0.02)

        #per WL
        for WL in WL_range:

            volume_sum = 0
            inundatie_sum = 0
            water_m2_sum = 0
            gras_m2_sum = 0
            akker_m2_sum = 0
            tuinbouw_m2_sum = 0
            bebouwing_m2_sum = 0
            water_m3_sum = 0
            gras_m3_sum = 0
            akker_m3_sum = 0
            tuinbouw_m3_sum = 0
            bebouwing_m3_sum = 0
            opp_pg_sum = 0

            # loop over peilgebieden
            for pg, df_pg in df_pol.groupby("peilgebied"):

                # pak alle waarden ≤ huidige WL
                df_pg_valid = df_pg[df_pg["WL"] <= WL]

                if df_pg_valid.empty:
                    continue

                # neem laatste (hoogste WL onder huidige WL)
                row = df_pg_valid.iloc[-1]

                volume_sum += row["volume_m3"]
                inundatie_sum += row["inundatie_m2"]
                water_m2_sum += row["water_m2"]
                gras_m2_sum += row["grasland_m2"]
                akker_m2_sum += row["akker_m2"]
                tuinbouw_m2_sum += row["tuinbouw_m2"]
                bebouwing_m2_sum += row["bebouwing_m2"]
                water_m3_sum += row["water_m3"]
                gras_m3_sum += row["grasland_m3"]
                akker_m3_sum += row["akker_m3"]
                tuinbouw_m3_sum += row["tuinbouw_m3"]
                bebouwing_m3_sum += row["bebouwing_m3"]
                opp_pg_sum += row["opp_pg_m2"]

            result_rows.append([
                ws,
                polder,
                WL,
                df_pol["opp_pldr_m2"].iloc[0],
                opp_pg_sum,
                volume_sum,
                inundatie_sum,
                water_m2_sum,
                gras_m2_sum,
                akker_m2_sum,
                tuinbouw_m2_sum,
                bebouwing_m2_sum,
                water_m3_sum,
                gras_m3_sum,
                akker_m3_sum,
                tuinbouw_m3_sum,
                bebouwing_m3_sum,
            ])

    df_out = pd.DataFrame(result_rows, columns=[
        "waterschap",
        "polder",
        "WL",
        "opp_pldr_m2",
        "opp_pg_m2",
        "volume_m3",
        "inundatie_m2",
        "water_m2",
        "grasland_m2",
        "akker_m2",
        "tuinbouw_m2",
        "bebouwing_m2",
        "water_m3",
        "grasland_m3",
        "akker_m3",
        "tuinbouw_m3",
        "bebouwing_m3"
    ])

    return df_out



### 2.2 Invoer: aggregeer dataframe naar polderniveau

Roep aggregate_to_polder_level functie op met genoemde dataframe in deel 2.1 en laat uitvoeren.


In [35]:
df_fokkesteeg_agg = aggregate_to_polder_level(df)
# df_per_polder.to_csv(r"D:\04_results\agv_results\agv_output_rvw.csv", index=False)
df_fokkesteeg_agg

,waterschap,polder,WL,opp_pldr_m2,opp_pg_m2,volume_m3,inundatie_m2,water_m2,grasland_m2,akker_m2,tuinbouw_m2,bebouwing_m2,water_m3,grasland_m3,akker_m3,tuinbouw_m3,bebouwing_m3
0,HDSR,Fokkesteeg,-1.400000e+00,3.308567e+06,3.446404e+05,1.747798e+01,73.00,1.00,0.00,0.00,0.00,69.25,0.001750,0.000000e+00,0.000000,0.000000,1.693546e+01
1,HDSR,Fokkesteeg,-1.380000e+00,3.308567e+06,3.446404e+05,2.172681e+02,10541.50,10417.75,0.00,0.00,0.00,121.00,197.582148,0.000000e+00,0.000000,0.000000,1.929354e+01
2,HDSR,Fokkesteeg,-1.360000e+00,3.308567e+06,3.446404e+05,4.339856e+02,11134.00,11005.50,0.00,0.00,0.00,125.50,411.778789,0.000000e+00,0.000000,0.000000,2.176003e+01
3,HDSR,Fokkesteeg,-1.340000e+00,3.308567e+06,3.446404e+05,6.622073e+02,11736.00,11603.00,0.00,0.00,0.00,129.75,637.372578,0.000000e+00,0.000000,0.000000,2.431106e+01
4,HDSR,Fokkesteeg,-1.320000e+00,3.308567e+06,3.446404e+05,9.012950e+02,12187.75,12049.00,0.00,0.00,0.00,135.50,874.003594,0.000000e+00,0.000000,0.000000,2.695850e+01
5,HDSR,Fokkesteeg,-1.300000e+00,3.308567e+06,3.446404e+05,1.148764e+03,12540.50,12397.75,0.00,0.00,0.00,139.25,1118.659375,0.000000e+00,0.000000,0.000000,2.969727e+01
6,HDSR,Fokkesteeg,-1.280000e+00,3.308567e+06,3.446404e+05,1.402978e+03,12889.00,12742.00,0.50,0.00,0.00,143.00,1369.978281,1.999960e-03,0.000000,0.000000,3.252027e+01
7,HDSR,Fokkesteeg,-1.260000e+00,3.308567e+06,3.446404e+05,1.664775e+03,13262.00,13109.00,0.50,0.00,0.00,148.75,1628.765469,1.199996e-02,0.000000,0.000000,3.544576e+01
8,HDSR,Fokkesteeg,-1.240000e+00,3.308567e+06,3.446404e+05,1.933152e+03,13564.00,13405.50,0.75,0.00,0.00,154.00,1894.021797,2.574999e-02,0.000000,0.000000,3.847750e+01
9,HDSR,Fokkesteeg,-1.220000e+00,3.308567e+06,3.446404e+05,2.207007e+03,13840.00,13675.25,1.00,0.00,0.00,160.00,2164.655938,4.450002e-02,0.000000,0.000000,4.160971e+01


# Deel 3 - Bereken RvW o.b.v. Ontwerpbui T

### 3.1 Genereer output .txt file + inundatieraster per polder 

**Doel**  
Deze stap vertaalt de eerder berekende volume-waterstandrelaties op polderniveau naar inundatiekaarten voor specifieke neerslagscenario's. Hiermee kan worden bepaald welke waterstand nodig is om een bepaalde hoeveelheid neerslag binnen een polder te bergen en welke delen van de polder daarbij inunderen.

**Achtergrond**
De inundatieberekeningen op polderniveau leveren een relatie tussen waterstand, inundatieoppervlak en bergingsvolume. Op basis van een opgegeven neerslaghoeveelheid wordt eerst het benodigde bergingsvolume bepaald:

```text
Bergingsvolume = Neerslaghoogte × Polderoppervlak
```

Vervolgens wordt binnen de beschikbare volume-waterstandrelatie gezocht naar de waterstand waarvan het berekende inundatievolume het dichtst bij dit benodigde bergingsvolume ligt.

**Werking**
- Bepalen van het benodigde bergingsvolume op basis van neerslag en polderoppervlak.
- Selecteren van de best passende waterstand uit de berekende volume-waterstandrelatie.
- Berekenen van de inundatiediepte door de geselecteerde waterstand te vergelijken met de maaiveldhoogte.


**Output**
Per polder worden de volgende bestanden gegenereerd:

- TXT-bestand met de geselecteerde waterstand en bijbehorende inundatiekenmerken.
- GeoTIFF-bestand met de berekende inundatiedieptes.

Daarnaast wordt per polder vastgelegd:

- waterschap
- polder
- polderoppervlak
- neerslagvolume
- geselecteerde waterstand
- inundatievolume


### 3.1 Achtergrondfunctie: Genereer output .txt file + inundatieraster per polder
Desbetreffende functie van boventstaande omschrijving

In [36]:
def find_closest_volume(df, waterschap, polder, target_volume):
    
    """
    Zoekt binnen een polder de waterstand waarvan het berekende
    inundatievolume het dichtst bij een opgegeven doelvolume ligt.
    Retourneert de volledige rij met resultaten.
    """
    df_pol = df[(df["waterschap"] == waterschap) &
                (df["polder"] == polder)].copy()   # ← copy() voorkomt ook de warning

    df_pol.loc[:, "abs_diff"] = (df_pol["volume_m3"] - target_volume).abs()

    return df_pol.loc[df_pol["abs_diff"].idxmin()]

def format_wl(value):
    
    """
    Formatteert een waterstand voor gebruik in bestandsnamen (is nu niet in gebruik maar optioneel).

    Voorbeelden:
    - -0,37   -> m0p37
    - -0.375  -> m0p38
    - 12      -> 12p00
    """
    s = str(value).strip()

    # Komma → punt
    s = s.replace(",", ".")

    # Naar float en afronden
    try:
        f = float(s)
    except ValueError:
        raise ValueError(f"Kan WL niet formatteren: {value}")

    f = round(f, 2)

    # Negatief teken veilig maken
    prefix = "m" if f < 0 else ""
    f = abs(f)

    # Punt vervangen door 'p'
    s = f"{f:.2f}".replace(".", "p")

    return f"{prefix}{s}"

from arcpy.sa import CreateConstantRaster, Raster

def process_polders_from_df_with_shapefile(
    df_per_polder,
    shapefile,
    polderpath_root,
    rain_mm,
 
    # ---- dataframe field mapping
    df_fld_waterschap,
    df_fld_polder,
 
    # ---- shapefile field mapping
    shp_fld_waterschap,
    shp_fld_polder,
    shp_fld_op_pldr,
 
    peil_type="zo",
    output_folder_name="resultaat"
):
    
    """
    Verwerkt inundatieresultaten op polderniveau voor een opgegeven
    neerslaggebeurtenis en genereert inundatiekaarten per polder.

    Voor iedere polder wordt op basis van het polderoppervlak en een
    opgegeven neerslaghoeveelheid een doelvolume bepaald. Vervolgens wordt
    uit de berekende volume-waterstandrelatie de waterstand geselecteerd
    waarvan het volume het dichtst bij dit doelvolume ligt.

    Per polder wordt een inundatieraster gemaakt waarin de waterdiepte
    zichtbaar is die nodig is om het berekende neerslagvolume te bergen.

    Returns
    -------
    Per polder worden de volgende bestanden aangemaakt:

    - TXT-bestand met de geselecteerde waterstand en bijbehorende
      inundatiekenmerken.
    - GeoTIFF-raster met de berekende inundatiediepte.

    Daarnaast wordt per polder bepaald:

    - waterschap
    - polder
    - polderoppervlak
    - neerslagvolume
    - geselecteerde waterstand (WL)
    - inundatievolume

    Werkwijze
    ---------
    1. Controleer of alle vereiste velden aanwezig zijn.
    2. Groepeer de invoergegevens per waterschap en polder.
    3. Lees het polderoppervlak uit de shapefile.
    4. Bereken het doelvolume op basis van de opgegeven neerslag:

       volume = neerslag (m) × polderoppervlak (m²)

    5. Zoek de waterstand waarvan het inundatievolume het dichtst
       bij het doelvolume ligt.
    6. Sla de geselecteerde resultaten op als TXT-bestand.
    7. Lees het peilraster van de polder.
    8. Bereken de inundatiediepte als:

       inundatie = waterstand − maaiveldhoogte

    9. Verwijder negatieve waarden zodat alleen inundatie wordt
       weergegeven.
    10. Sla het inundatieraster op als GeoTIFF.

    Opmerkingen
    -----------
    De functie gebruikt eerder berekende volume-waterstandrelaties op
    polderniveau. Hierdoor kan voor verschillende neerslagscenario's
    snel een inundatiekaart worden gegenereerd zonder de volledige
    volumeberekening opnieuw uit te voeren.
    """

    arcpy.CheckOutExtension("Spatial")
 
    # --------------------------------------------------
    # Validate dataframe schema
    for fld in [df_fld_waterschap, df_fld_polder]:
        if fld not in df_per_polder.columns:
            raise KeyError(
                f"Kolom '{fld}' ontbreekt in df_per_polder"
            )
 
    # --------------------------------------------------
    # Validate shapefile schema
    shp_fields = [f.name for f in arcpy.ListFields(shapefile)]
    for fld in [shp_fld_waterschap, shp_fld_polder, shp_fld_op_pldr]:
        if fld not in shp_fields:
            raise KeyError(
                f"Veld '{fld}' ontbreekt in shapefile"
            )
 
    # --------------------------------------------------
    # Loop over polders (bron van waarheid = df + folderstructuur)
    grouped = df_per_polder.groupby(
        [df_fld_waterschap, df_fld_polder]
    )
 
    for (waterschap, polder), df_pol in grouped:
 
        print(f"--- Verwerk {waterschap} / {polder} ---")
 
        ws_norm = maak_veilige_naam(waterschap, target="filesystem")
        pol_norm = maak_veilige_naam(polder, target="filesystem")
        
        
        polder_path = os.path.join(
            polderpath_root,
            ws_norm,
            pol_norm
        )

        # --------------------------------------------------
        # 🔗 Lees + sommeer oppervlak uit shapefile
        waterschap_sql = str(waterschap).replace("'", "''")
        polder_sql = str(polder).replace("'", "''")
        
        where = (
            f"{shp_fld_waterschap} = '{waterschap_sql}' AND "
            f"{shp_fld_polder} = '{polder_sql}'"
        )
 
        opp_pldr_m2 = 0.0
        count = 0
 
        with arcpy.da.SearchCursor(
            shapefile,
            [shp_fld_op_pldr],
            where_clause=where
        ) as cursor:
            for (opp,) in cursor:
                if opp is not None:
                    opp_pldr_m2 += float(opp)
                    count += 1
 
        if count == 0:
            raise ValueError(
                f"Geen oppervlak gevonden in shapefile voor "
                f"{waterschap} / {polder}"
            )
 
        if opp_pldr_m2 <= 0:
            raise ValueError(
                f"Ongeldig polderoppervlak voor "
                f"{waterschap} / {polder}: {opp_pldr_m2}"
            )
 
        print(
            f"    • oppervlak uit shapefile: "
            f"{count} deelgebieden, totaal = {int(opp_pldr_m2)} m²"
        )
 
        # --------------------------------------------------
        # Target volume from rainfall
        target_volume = (rain_mm / 1000.0) * opp_pldr_m2
 
        result = find_closest_volume(
            df_pol,
            waterschap,
            polder,
            target_volume
        )
 
        WL_raw = result["WL"]
        WL_fmt = format_wl(WL_raw)
 
        # --------------------------------------------------
        # Paths
 
        out_dir = os.path.join(polder_path, output_folder_name)
        os.makedirs(out_dir, exist_ok=True)
 
        # --------------------------------------------------
        # Write TXT
        rain_tag = f"bui{int(rain_mm)}mm"

        txt_path = os.path.join(
            out_dir,
            f"{ws_norm}_{pol_norm}_{rain_tag}.txt"
        )
 
        result.to_frame().T.to_csv(
            txt_path,
            sep="\t",
            index=False
        )
 
        # --------------------------------------------------
        # Raster calculation: WL – peil
        peil_raster = os.path.join(
            polder_path,
            f"{ws_norm}_{pol_norm}_peil_{peil_type}.tif"
        )
         
        if not arcpy.Exists(peil_raster):
            raise FileNotFoundError(
                f"Peilraster ontbreekt: {peil_raster}"
            )
 
        with arcpy.EnvManager(
            extent=peil_raster,
            cellSize=peil_raster,
            snapRaster=peil_raster
        ):
            inundatie = (
                arcpy.sa.Float(WL_raw)
                - (arcpy.sa.Raster(peil_raster) / 1000)
            )
 
            inundatie = SetNull(inundatie <= 0, inundatie)
 
            out_raster = os.path.join(
                out_dir,
                f"{ws_norm}_{pol_norm}_{rain_tag}.tif"
            )
 
            inundatie.save(out_raster)
 
        print(
            f"  ✓ WL={WL_raw}, "
            f"neerslag={rain_mm} mm, "
            f"volume≈{int(target_volume)} m³"
        )


### 3.1 Invoer: genereer output .txt file + inundatieraster per polder 
Voor het genereren van inundatiekaarten moeten de volgende gegevens worden opgegeven:
- **df_per_polder**: DataFrame met de geaggregeerde volume-waterstandrelaties op polderniveau.
- **shapefile**: Feature class of shapefile met de poldergeometrieën en oppervlakten.
- **polderpath_root**: Hoofdmap waarin de polderresultaten en rasters zijn opgeslagen.
- **rain_mm**: Neerslaghoeveelheid (mm) waarvoor de inundatiekaart wordt berekend.
- **df_fld_waterschap**: Kolomnaam in de DataFrame met de waterschapsnaam.
- **df_fld_polder**: Kolomnaam in de DataFrame met de poldernaam.
- **shp_fld_waterschap**: Veldnaam in de shapefile met de waterschapsnaam.
- **shp_fld_polder**: Veldnaam in de shapefile met de poldernaam.
- **shp_fld_op_pldr**: Veldnaam met de oppervlakte van de polderdelen. Indien een polder uit meerdere polygonen bestaat worden deze oppervlakten automatisch gesommeerd.
- **peil_type**: Type peilraster dat wordt gebruikt, bijvoorbeeld zomerpeil (`zo`) of winterpeil (`wi`).
 
**Voorbeeldconfiguratie**
```text
Neerslagscenario : 92 mm
Waterschapsveld : Waterschap
Polderveld : Naam_1
Oppervlakteveld : Shape_Area
Peiltype : zomerpeil (zo)
Resultaatlocatie: D:\04_results
```


In [37]:
process_polders_from_df_with_shapefile(
    df_per_polder=df_fokkesteeg_agg,
    shapefile=r"D:\04_results\hdsr\fokkesteeg\fokkesteeg.gdb\rekengebied_fokkesteeg", #vul hier het rekengebied in
    polderpath_root=r"D:\04_results",
    rain_mm=92,

    # ---- dataframe field mapping
    df_fld_waterschap="waterschap",
    df_fld_polder="polder",

    # ---- shapefile field mapping
    shp_fld_waterschap="Waterschap",
    shp_fld_polder="Naam_1",
    shp_fld_op_pldr="Shape_Area", #De area van de peilvakken [wordt gesommeerd]

    peil_type="zo"
)

--- Verwerk HDSR / Fokkesteeg ---
    • oppervlak uit shapefile: 10 deelgebieden, totaal = 3308434 m²
  ✓ WL=0.740000000000002, neerslag=92 mm, volume≈304375 m³


### 3.2 Inlezen van resultaten en berekenen van RvW 

**Doel**

Deze stap leest de resultaten van een neerslagscenario per polder in en berekent de RVW-indicator (Ruimte voor Water). De RVW-indicator geeft inzicht in de mate waarin economisch of maatschappelijk relevant landgebruik wordt getroffen door inundatie.

**Context**

Voor iedere polder is in een eerdere stap een resultatenbestand opgeslagen met inundatieoppervlaktes en inundatievolumes per landgebruikstype. Deze functie verzamelt deze resultaten en berekent daaruit een samenvattende RVW-indicator.

Bij de RVW-berekening wordt uitsluitend gekeken naar landgebruikstypen waarbij inundatie potentieel kan leiden tot schade of hinder:

- grasland
- akkerbouw
- hoogwaardig land- en tuinbouw
- bebouwing

Open water wordt hierbij buiten beschouwing gelaten.

**Output**

Per polder worden de volgende aanvullende indicatoren berekend:

- **rvw_m2**: totaal overstroomd oppervlak van grasland, akker, tuinbouw en bebouwing (m²)
- **rvw_m3**: totaal inundatievolume op grasland, akker, tuinbouw en bebouwing (m³)
- **rvw_%**: percentage van het totale polderoppervlak dat onder RVW valt

Daarnaast blijven alle eerder berekende inundatiekenmerken beschikbaar.


### 3.2 Achtergrondfunctie: Inlezen van resultaten en berekenen van RvW
Desbetreffende functie van boventstaande omschrijving

In [61]:
def lees_resultaten(
    df_per_polder,
    root_folder,
    rain_mm
):
    
    """
    Leest de resultaten van een neerslagscenario per polder in en
    berekent aanvullende RVW-indicatoren.

    Voor iedere polder wordt het eerder opgeslagen resultatenbestand
    (TXT) opgezocht en ingelezen. Vervolgens worden de oppervlaktes en
    volumes van de relevante landgebruikstypen samengevoegd tot één
    RVW-waarde.

    Returns
    -------
    pandas.DataFrame

    DataFrame met de ingelezen resultaten per polder, aangevuld met:

    - rvw_m2: totaal overstroomd oppervlak van grasland, akker,
      tuinbouw en bebouwing (m²)
    - rvw_m3: totaal inundatievolume op grasland, akker,
      tuinbouw en bebouwing (m³)
    - rvw_%: percentage van het polderoppervlak dat onder RVW valt

    Werkwijze
    ---------
    1. Groepeer de invoergegevens per waterschap en polder.
    2. Zoek voor iedere polder het resultatenbestand van het
       opgegeven neerslagscenario.
    3. Lees de resultatenbestanden in als pandas DataFrame.
    4. Voeg alle resultaten samen tot één totaaloverzicht.
    5. Bereken het totale RVW-oppervlak door de oppervlaktes van:
       - grasland
       - akker
       - tuinbouw
       - bebouwing
       op te tellen.
    6. Bereken het totale RVW-volume voor dezelfde landgebruikstypen.
    7. Bereken het RVW-percentage ten opzichte van het totale
       polderoppervlak.
    8. Retourneer de verrijkte DataFrame.

    Opmerkingen
    -----------
    Wateroppervlak wordt niet meegenomen in de RVW-berekening.
    De RVW-indicator richt zich uitsluitend op de landgebruikstypen
    waar inundatie tot schade of hinder kan leiden.
    """

    rain_tag = f"bui{int(rain_mm)}mm"

    dfs = []

    grouped = df_per_polder.groupby(
        ["waterschap", "polder"]
    )

    for (waterschap, polder), _ in grouped:

        ws_norm = maak_veilige_naam(
            waterschap,
            target="filesystem"
        )

        pol_norm = maak_veilige_naam(
            polder,
            target="filesystem"
        )

        result_dir = (
            Path(root_folder)
            / ws_norm
            / pol_norm
            / "resultaat"
        )

        matches = list(
            result_dir.glob(
                f"{ws_norm}_{pol_norm}_{rain_tag}*.txt"
            )
        )

        if not matches:
            print(f"Niet gevonden: {result_dir}")
            continue

        dfs.append(
            pd.read_csv(matches[0], sep="\t")
        )

    df_all = pd.concat(
        dfs,
        ignore_index=True
    )
    
    df_all = df_all.drop(columns =["peilgebied"], 
                        errors="ignore"
                        )

    # ------------------------------
    # RVW-berekeningen

    df_all["rvw_m2"] = df_all[
        [
            "grasland_m2",
            "akker_m2",
            "tuinbouw_m2",
            "bebouwing_m2"
        ]
    ].sum(axis=1)

    df_all["rvw_m3"] = df_all[
        [
            "grasland_m3",
            "akker_m3",
            "tuinbouw_m3",
            "bebouwing_m3"
        ]
    ].sum(axis=1)

    df_all["rvw_m2_%"] = (
        df_all["rvw_m2"]
        / df_all["opp_pldr_m2"]
    ) * 100

    
    df_all["rvw_m3_%"] = (
        df_all["rvw_m3"] 
        / df_all["volume_m3"]
    ) * 100
    
    df_all["sloot_%"] = (
        df_all["water_m3"]
        / df_all["volume_m3"]
    ) * 100
    
    df_all["water_m2_%"] = (
        df_all["water_m2"]
        / df_all["opp_pldr_m2"]
    ) * 100
    
    df_all["opp_pldr_ha"] = (
        df_all["opp_pldr_m2"] / 10000)
    
    df_all["gem_inun_m"] = ((df_all["grasland_m3"] + df_all["akker_m3"] + df_all["tuinbouw_m3"] + df_all["bebouwing_m3"]) / 
                            (df_all["grasland_m2"] + df_all["akker_m2"] + df_all["tuinbouw_m2"] + df_all["bebouwing_m2"])) * 100
    
    return df_all


### 3.2 Invoer: Inlezen van resultaten en berekenen van RvW


- **df_per_polder**: DataFrame met de geaggregeerde inundatieresultaten op polderniveau.
- **root_folder**: Hoofdmap waarin de resultaten per waterschap en polder zijn opgeslagen.
- **rain_mm**: Neerslagscenario waarvoor de resultaten moeten worden ingelezen.

**Resulteert in:**
De functie levert één verrijkte resultatentabel op waarin per polder zowel de inundatieresultaten als de afgeleide RVW-indicatoren beschikbaar zijn voor verdere analyse en rapportage.

In [40]:
# df_agv_polder  = pd.read_csv(r"D:\04_results\agv_results\agv_output_rvw.csv")

df_fokkesteeg = lees_resultaten(
    df_per_polder=df_fokkesteeg_agg,
    root_folder=r"D:\04_results",
    rain_mm=92
)

# output_csv = r"D:\04_results\agv_results\df_agv_bui92mm.csv"

# df_agv_rvw.to_csv(
#     output_csv,
#     index=False
# )
pd.set_option("display.max_rows", None)
df_fokkesteeg

,waterschap,polder,WL,opp_pldr_m2,opp_pg_m2,volume_m3,inundatie_m2,water_m2,grasland_m2,akker_m2,tuinbouw_m2,bebouwing_m2,water_m3,grasland_m3,akker_m3,tuinbouw_m3,bebouwing_m3,abs_diff,rvw_m2,rvw_m3,rvw_m2_%,rvw_m3_%,sloot_%,water_m2_%,opp_pldr_ha,gem_inun_m
0,HDSR,Fokkesteeg,0.74,3.308567e+06,3.309050e+06,300842.065925,810357.0,250299.0,179762.75,67.5,0.0,379946.25,182052.588522,45426.648161,17.064754,0.0,73260.498132,3533.888307,559776.5,118704.211048,16.919,39.457318,60.514339,7.565178,330.85673,21.205644


In [65]:
df_agv_polder  = pd.read_csv(r"D:\04_results\agv_results\agv_output_rvw.csv")

df_agv_rvw = lees_resultaten(
    df_per_polder=df_agv_polder,
    root_folder=r"D:\04_results",
    rain_mm=92
)

# output_csv = r"D:\04_results\agv_results\df_agv_bui92mm.csv"

# df_agv_rvw.to_csv(
#     output_csv,
#     index=False
# )
pd.set_option("display.max_rows", None)
df_agv_rvw

,waterschap,polder,WL,opp_pldr_m2,opp_pg_m2,volume_m3,inundatie_m2,water_m2,grasland_m2,akker_m2,tuinbouw_m2,bebouwing_m2,water_m3,grasland_m3,akker_m3,tuinbouw_m3,bebouwing_m3,abs_diff,rvw_m2,rvw_m3,rvw_m2_%,rvw_m3_%,sloot_%,water_m2_%,opp_pldr_ha,gem_inun_m
0,AGV,'s-Gravelandsche Polder,4.800000e-01,8.480336e+06,6.034040e+06,7.872289e+05,2640746.00,516060.25,1893194.25,57907.75,65181.00,108402.75,3.079263e+05,4.287606e+05,1.822769e+04,15698.804262,16615.481749,7037.984781,2124685.75,4.793026e+05,25.054264,60.884785,39.115215,6.085375,848.033601,22.558754
1,AGV,'s-Gravelandsche vaartboezem,1.400000e-01,7.363108e+06,7.308184e+06,6.648993e+05,1773460.50,1349733.00,400225.25,6056.50,0.00,17445.75,5.709606e+05,8.951812e+04,7.888102e+02,0.000000,3631.753768,12506.654755,423727.50,9.393869e+04,5.754737,14.128258,85.871742,18.331023,736.310793,22.169599
2,AGV,Aetsveldse Polder Oost,-1.440000e+00,8.547763e+06,8.345713e+06,7.660771e+05,3130653.25,505734.50,2468455.00,124809.00,15248.50,16406.25,2.438994e+05,4.841668e+05,3.299661e+04,1836.687502,3177.540013,20317.088059,2624918.75,5.221777e+05,30.708839,68.162547,31.837453,5.916571,854.776299,19.893098
3,AGV,Aetsveldse Polder west,-1.450000e+00,2.777844e+06,2.729345e+06,2.458466e+05,1076440.00,178572.75,888981.75,190.00,0.00,8695.50,9.265632e+04,1.521241e+05,1.797850e+01,0.000000,1048.213251,9715.059613,897867.25,1.531903e+05,32.322445,62.311334,37.688666,6.428465,277.784448,17.061577
4,AGV,Aetsveldse Polder west (Driemond),-9.300000e-01,6.741670e+04,6.741670e+04,3.451658e+03,11255.25,999.75,5906.00,0.00,0.00,4349.50,8.680522e+02,2.163673e+03,0.000000e+00,0.000000,419.932498,2750.678496,10255.50,2.583606e+03,15.212106,74.851151,25.148849,1.482941,6.741670,25.192392
5,AGV,Atekpolder,-1.000000e+00,2.587904e+04,2.587904e+04,1.222137e+03,3809.00,697.50,1586.25,0.00,0.00,1525.25,5.032440e+02,5.122383e+02,0.000000e+00,0.000000,206.654750,1158.734262,3111.50,7.188930e+02,12.023246,58.822620,41.177380,2.695232,2.587904,23.104387
6,AGV,B.O.B.M.-polder en Buitendijken tussen Muiderberg,-5.200000e-01,2.684487e+06,2.644617e+06,2.550460e+05,970128.75,222543.75,727245.50,13710.50,0.00,6629.00,1.087550e+05,1.437135e+05,1.046787e+03,0.000000,1530.595250,8073.123865,747585.00,1.462909e+05,27.848335,57.358655,42.641345,8.289991,268.448725,19.568468
7,AGV,BP Huis Te Vraag,-8.000000e-02,4.193783e+04,4.193783e+04,3.997990e+03,14908.50,3636.50,9241.50,0.00,0.00,2030.50,1.512124e+03,2.287911e+03,0.000000e+00,0.000000,197.955251,139.710071,11272.00,2.485866e+03,26.877882,62.177896,37.822104,8.671169,4.193783,22.053462
8,AGV,Baambrugge Oostzijds,-1.810000e+00,7.252899e+06,7.370530e+06,6.394840e+05,3207400.25,650061.50,2532098.50,5188.50,0.00,20051.75,3.065543e+05,3.302192e+05,4.840878e+02,0.000000,2226.409258,27782.712624,2557338.75,3.329297e+05,35.259538,52.062242,47.937758,8.962781,725.289912,13.018600
9,AGV,Baambrugge Oostzijds (west),-1.930000e+00,1.393604e+06,8.189574e+05,1.253611e+05,524793.00,80719.75,427770.25,10.25,0.00,16292.75,4.540903e+04,7.825429e+04,5.832500e-01,0.000000,1697.208755,2850.498524,444073.25,7.995209e+04,31.865085,63.777422,36.222578,5.792156,139.360448,18.004256


### 3.3 Controleer missende polders 

**Doel**  
Optioneel kan je de originele shapefile die je hebt gebruikt om de rekengebieden de genereren hier invoeren om te checken welke polders niet in de dataset voorkomen.

In [1]:
import arcpy
import pandas as pd

def find_missing_polders(
    shapefile,
    df_all,
    shp_fld_waterschap="Waterschap",
    shp_fld_polder="Naam_1"
):

    shp_set = {
        (str(ws).strip(), str(pol).strip())
        for ws, pol in arcpy.da.SearchCursor(
            shapefile,
            [shp_fld_waterschap, shp_fld_polder]
        )
    }

    df_set = {
        (str(ws).strip(), str(pol).strip())
        for ws, pol in zip(
            df_all["waterschap"],
            df_all["polder"]
        )
    }

    missing = shp_set - df_set

    return pd.DataFrame(
        sorted(missing),
        columns=["waterschap", "polder"]
    )

In [3]:
df_hhr_all = pd.read_csv(r"D:\04_results\hhr_results\df_hhr_bui92mm.csv")

df_missing = find_missing_polders(
    shapefile=r"C:\Users\Senden02\OneDrive - Waternet Amsterdam\Documenten\ArcGIS\Projects\RvW-DPCH\rvw_tool\01_src\03_data_processing\ahn_merger\ahn_merger.gdb\afvoergebieden_compleet_correct_dp_hout_hhr",
    df_all=df_hhr_all
)

df_missing

,waterschap,polder
0,Rijnland,De Verdolven Landen
1,Rijnland,Haarlemmermeerpolder
2,Rijnland,Hogergelegen Santpoort
3,Rijnland,Inmaling Duinland
4,Rijnland,Inmaling Duinrell
5,Rijnland,Kadebuurt
6,Rijnland,Kikkerpolder
7,Rijnland,Landgoed De Paauw
8,Rijnland,Landgoed de Wittenburg
9,Rijnland,Lentevreugd


# Deel 4 - Data-analyse

### 4.1 Histogrammen

In [65]:
import matplotlib.pyplot as plt

def plot_histogram(
    df,
    col,
    title=None,
    xlabel=None,
    bins=100,
    figsize=(6, 5),
    xmin=None,
    xmax=None
):

    fig, ax = plt.subplots(figsize=figsize)

    df[col].hist(
        bins=bins,
        edgecolor="black",
        ax=ax
    )

    ax.set_title(title or col)
    ax.set_xlabel(xlabel or col)
    ax.set_ylabel("Aantal polders")

    if xmin is not None or xmax is not None:
        ax.set_xlim(xmin, xmax)

    plt.tight_layout()
    plt.show()

In [67]:
plot_histogram(
    df_hhr_rvw,
    "abs_diff",
    title="abs_diff (m3) HHR",
    xlabel="abs_diff (m3)",
    xmin=0,
    xmax=650000
)

In [72]:
import matplotlib.pyplot as plt

def plot_histograms_waterschappen(
    dfs,
    namen,
    col,
    bins=20,
    figsize=(14, 10),
    xmin=None,
    xmax=None,
    sharex=True,
    sharey=True
):
    """
    Plot dezelfde variabele voor meerdere waterschappen.

    Parameters
    ----------
    dfs : list
        Lijst met dataframes (of None).
    namen : list
        Namen van de waterschappen.
    col : str
        Te plotten kolom.
    """

    fig, axes = plt.subplots(
        2,
        2,
        figsize=figsize,
        sharex=sharex,
        sharey=sharey
    )

    axes = axes.flatten()

    for ax, df, naam in zip(axes, dfs, namen):

        if df is None:
            ax.set_title(naam)
            ax.text(
                0.5,
                0.5,
                "Nog niet beschikbaar",
                ha="center",
                va="center",
                fontsize=12
            )
            continue

        df[col].hist(
            bins=bins,
            edgecolor="black",
            ax=ax
        )

        ax.set_title(naam)
        ax.set_xlabel(col)
        ax.set_ylabel("Aantal polders")

        if xmin is not None or xmax is not None:
            ax.set_xlim(xmin, xmax)

    plt.tight_layout()
    plt.show()

In [77]:
plot_histograms_waterschappen(
    dfs=[
        None,          # HHNK rekent nog
        df_hhr_rvw,
        df_hdsr_rvw,
        df_agv_rvw
    ],
    namen=[
        "HHNK",
        "HHR",
        "HDSR",
        "AGV"
    ],
    col="rvw_m2_%",
    bins=20,
    xmin=0,
    xmax=90
)

In [161]:
import matplotlib.pyplot as plt

def plot_waterschap_dashboard(
    df,
    waterschap_naam,
    bins=30
):
    """
    3x3 dashboard met histogrammen voor één waterschap.
    """

    fig, axes = plt.subplots(
        2,
        3,
        figsize=(15, 12)
    )
    
    axes = axes.flatten()

    variabelen = [
        ("sloot_%", "% buivolume in sloot (m3) t.o.v. totale bui (m3)"),
        ("rvw_m3_%", "% RvW op maaivled (m3) t.o.v. totale bui (m3)"),
        ("rvw_m2_%", "% RvW op maaiveld (m2) t.o.v. polderoppervlak (m2)"),
#         ("peilstijging_cm", "Peilstijging (cm)"),
        ("water_m2_%", "% water (m2) t.o.v. polderoppervlak (m2)"),
        ("opp_pldr_ha", "Polderoppervlak (ha)"),
#         ("bebouwing_pct", "% bebouwing"),
        ("gem_inun_m", "Gem. inundatiediepte (cm)"),
#         ("p90_diepte_cm", "P90 inundatiediepte (cm)")
    ]

    for ax, (kolom, titel) in zip(axes, variabelen):

        if kolom not in df.columns:
            ax.text(
                0.5,
                0.5,
                f"Kolom ontbreekt:\n{kolom}",
                ha="center",
                va="center"
            )
            ax.set_title(titel)
            continue

        df[kolom].hist(
            bins=bins,
            edgecolor="black",
            ax=ax
        )

        ax.set_title(titel)
        ax.set_ylabel("Aantal polders")

    fig.suptitle(
        f"Waterschap {waterschap_naam}",
        fontsize=16
    )

    plt.tight_layout()
    plt.show()

In [168]:
plot_waterschap_dashboard(
    df_hdsr_rvw,
    "HDSR"
)
